# 科产融合案例库审阅复算
## 结论与范围
769条记录分为181个机构与588个资助项目；机构日期有证据46例。来源和观察层级明显影响模式构成。本文仅进行数据诊断，不估计总体准确率、政策因果或国家绩效。

## 方法
冻结版本2026.10.14-r2；第一版保持不变。按记录层级和采集批次计算覆盖与构成，以逐一剔除批次检查敏感性。单来源统计只覆盖主要事实叙述所引用的case.evidence，不含日期专用来源。

In [1]:
from pathlib import Path
import json, hashlib
from collections import Counter
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'config/open_innovation_models.json').exists())
out=root/'analysis/innovation_research_20261014_r2'
book=json.loads((out/'data/cases_nested.json').read_text(encoding='utf-8'))
old=json.loads((root/'analysis/innovation_research_20261014/data/cases_nested.json').read_text(encoding='utf-8'))
cases=book['cases']; institutions=[c for c in cases if c['research_annotation']['grain']=='platform']
print({'cases':len(cases),'institutions':len(institutions),'projects':len(cases)-len(institutions)})
print([{'case_id':c['id'],'country':c['research_annotation'].get('country'),'origin':c['timeline']['origin_year']} for c in institutions[:5]])

{'cases': 769, 'institutions': 181, 'projects': 588}
[{'case_id': 'oi_artc', 'country': 'SG', 'origin': 2012}, {'case_id': 'oi_gba', 'country': None, 'origin': None}, {'case_id': 'oi_nedo_lyon', 'country': None, 'origin': None}, {'case_id': 'oi_cerc', 'country': None, 'origin': None}, {'case_id': 'oi_clean_hydrogen', 'country': None, 'origin': None}]


## 核查结果
### 保留旧记录与主键

In [2]:
current={c['id']:c for c in cases}
assert len(current)==len(cases)
ignored={'research_annotation','timeline','organization_tags','organization_participant_tags','organization_basis','developments'}
for case in old['cases']:
    assert case['id'] in current
    for field,value in case.items():
        if field not in ignored:assert current[case['id']][field]==value,(case['id'],field)
sources={s['id']:s for s in book['sources']}
assert len(sources)==len(book['sources'])==len({s['url'] for s in book['sources']})
assert all(ref['source_id'] in sources for c in cases for ref in c['evidence'])
print({'retained_case_facts':len(old['cases']),'new_cases':len(cases)-len(old['cases']),'sources':len(sources)})

{'retained_case_facts': 761, 'new_cases': 8, 'sources': 811}


### 分层时间缺失率与单来源依赖

In [3]:
rows=[]
for grain in ('platform','project'):
    subset=[c for c in cases if c['research_annotation']['grain']==grain]
    known=sum(c['timeline']['origin_year'] is not None for c in subset)
    rows.append({'grain':grain,'denominator':len(subset),'dated':known,'coverage':known/len(subset)})
single=sum(len({e['source_id'] for e in c['evidence']})==1 for c in cases)
print(rows)
print({'single_narrative_source':single,'denominator':len(cases),'fraction':single/len(cases)})
assert rows[0]['dated']==46 and rows[1]['dated']==588

[{'grain': 'platform', 'denominator': 181, 'dated': 46, 'coverage': 0.2541436464088398}, {'grain': 'project', 'denominator': 588, 'dated': 588, 'coverage': 1.0}]
{'single_narrative_source': 752, 'denominator': 769, 'fraction': 0.9778933680104032}


### 观察层级与来源剔除敏感性

In [4]:
diagnostics=book['research']['review_diagnostics']
models=book['analysis']['archetypes']
for model in models:
    full=sum(c['research_annotation']['primary_model']==model['id'] for c in cases)/len(cases)
    platform=sum(c['research_annotation']['primary_model']==model['id'] for c in institutions)/len(institutions)
    print(model['id'],{'pooled_share':round(full,4),'institution_share':round(platform,4),'difference_percentage_points':round(100*(platform-full),2)})
for row in diagnostics['model_sensitivity']:
    subset=[c for c in cases if (row['grain']=='all' or c['research_annotation']['grain']==row['grain']) and c['research_annotation']['cohort']!=row['omitted_cohort']]
    assert len(subset)==row['denominator']
    assert sum(c['research_annotation']['primary_model']==row['model'] for c in subset)==row['count']
print('All source-omission model counts reconciled independently')

venture {'pooled_share': 0.0117, 'institution_share': 0.0497, 'difference_percentage_points': 3.8}
mission {'pooled_share': 0.2627, 'institution_share': 0.0663, 'difference_percentage_points': -19.64}
intermediary {'pooled_share': 0.0637, 'institution_share': 0.2707, 'difference_percentage_points': 20.7}
consortium {'pooled_share': 0.5735, 'institution_share': 0.2376, 'difference_percentage_points': -33.59}
pilot {'pooled_share': 0.0377, 'institution_share': 0.1602, 'difference_percentage_points': 12.25}
cluster {'pooled_share': 0.0234, 'institution_share': 0.0994, 'difference_percentage_points': 7.6}
capacity {'pooled_share': 0.0221, 'institution_share': 0.0939, 'difference_percentage_points': 7.18}
rules {'pooled_share': 0.0052, 'institution_share': 0.0221, 'difference_percentage_points': 1.69}
All source-omission model counts reconciled independently


### 要素组合与非置信区间范围

In [5]:
for row in diagnostics['factor_sensitivity']:
    for estimate in row['source_cohort_results']:
        subset=[c for c in institutions if c['research_annotation']['cohort']!=estimate['omitted_cohort']]
        both=sum(row['first'] in c['factors'] and row['second'] in c['factors'] for c in subset)
        union=sum(row['first'] in c['factors'] or row['second'] in c['factors'] for c in subset)
        assert both==estimate['count'] and union==estimate['union']
        assert estimate['jaccard']==(both/union if union else None)
    if row['first'] not in ('knowledge','market') and row['second'] not in ('knowledge','market'):
        print({k:row[k] for k in ('first','second','count','jaccard','leave_one_cohort_out_min','leave_one_cohort_out_max')})

{'first': 'talent', 'second': 'capital', 'count': 7, 'jaccard': 0.08235294117647059, 'leave_one_cohort_out_min': 0.0625, 'leave_one_cohort_out_max': 0.11475409836065574}
{'first': 'talent', 'second': 'infrastructure', 'count': 23, 'jaccard': 0.21100917431192662, 'leave_one_cohort_out_min': 0.16666666666666666, 'leave_one_cohort_out_max': 0.27058823529411763}
{'first': 'talent', 'second': 'data', 'count': 1, 'jaccard': 0.013157894736842105, 'leave_one_cohort_out_min': 0.0, 'leave_one_cohort_out_max': 0.019230769230769232}
{'first': 'capital', 'second': 'infrastructure', 'count': 7, 'jaccard': 0.07865168539325842, 'leave_one_cohort_out_min': 0.06779661016949153, 'leave_one_cohort_out_max': 0.09523809523809523}
{'first': 'capital', 'second': 'data', 'count': 0, 'jaccard': 0.0, 'leave_one_cohort_out_min': 0.0, 'leave_one_cohort_out_max': 0.0}
{'first': 'infrastructure', 'second': 'data', 'count': 7, 'jaccard': 0.0945945945945946, 'leave_one_cohort_out_min': 0.05172413793103448, 'leave_one_

### 固定文件校验

In [6]:
manifest=json.loads((out/'manifest.json').read_text(encoding='utf-8'))
for f in manifest['files']:
    data=(out/f['path']).read_bytes()
    assert hashlib.sha256(data).hexdigest()==f['sha256'] and len(data)==f['bytes']
print({'frozen_data_files':len(manifest['files']),'checksum_status':'passed'})

{'frozen_data_files': 16, 'checksum_status': 'passed'}


## 使用判断
可用于来源可追溯的案例检索和分层描述研究。日期缺失、单一来源、实体粒度差异以及未验证的编码仍限制历史和绩效推断。没有独立双人核验，不报告准确率。